# **XGBoost** классификация


это реализация градиентного бустинга над деревьями решений с регуляризацией, которая последовательно добавляет новые деревья, уменьшая ошибку предыдущих и тем самым строя сильную модель из множества слабых.


## Предобработка

In [ ]:
import sys, os
sys.path.append(os.path.abspath("../.."))

In [ ]:
import pandas as pd
from data_preprocessing.DataForModel import build_target, split_data, preprocess_dataset

In [11]:
name = input("Введите имя файла большими буквами: ")
df: pd.DataFrame = pd.read_csv(f"/Users/side/Desktop/Trading Chaos AI/df/clean_df/{name}.csv")

In [12]:
df = preprocess_dataset(df)

In [13]:
df = build_target(df, h=20, target_type="classification")

y = df["GoodTrade"]

In [15]:
X_train, X_test, y_train, y_test = split_data(df, target="GoodTrade", val_size=0.1, test_size=0.2, split_type="train_test")

print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")

X_train: (29088, 33), X_test: (7272, 33)


## Обучение

In [16]:
# Обучаем XGBoost для классификации
model = xgb.XGBClassifier(use_label_encoder=False, eval_metric="logloss")

# Тренировка модели
model.fit(X_train, y_train)

# Прогнозируем на тестовой выборке
y_pred = model.predict(X_test)

# Оценка модели
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")

/Users/side/Desktop/Trading Chaos AI/chaos_env/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [15:12:51] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


NameError: name 'accuracy_score' is not defined

настройка баланса классов

In [ ]:
# баланс классов
pos = (y_train == 1).sum()
neg = (y_train == 0).sum()
scale_pos_weight = neg / pos if pos > 0 else 1.0
print(f"pos={pos}, neg={neg}, scale_pos_weight={scale_pos_weight:.2f}")


Модель

In [ ]:
model = xgb.XGBClassifier(
    objective='binary:logistic',
    n_estimators=1000,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.7,
    colsample_bytree=0.7,
    reg_lambda=2.0,
    scale_pos_weight=scale_pos_weight,
    tree_method="hist",
    eval_metric="auc",
)

model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],   # будем смотреть AUC на тесте
    verbose=50                     # печатать лог каждые 50 итераций
)



In [ ]:
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix

# вероятности того, что сделка хорошая
proba_test = model.predict_proba(X_test)[:, 1]

# можно взять порог 0.5 (потом поиграемся)
y_pred = (proba_test >= 0.5).astype(int)

print("AUC на тесте:", roc_auc_score(y_test, proba_test))
print()
print("Отчёт по классификации:")
print(classification_report(y_test, y_pred))

print("Матрица ошибок:")
print(confusion_matrix(y_test, y_pred))


In [ ]:
import pandas as pd
import numpy as np

importances = model.feature_importances_
fi = pd.DataFrame({'feature': feature_cols, 'importance': importances})
fi = fi.sort_values('importance', ascending=False)
fi.head(20)
